# Data prep 

Prepping background data for LCA tool. 
- `../data/processed/unit_burdens.csv` - unit EF burdens (e.g. kgCO2) of 
    - material production and transportation to factory for multiple biobased materials (kgCO2/kg)
    - machine wear, based on ecoinvent dataset for "market for industrial machine" (kgCO2/kg)
    - energy use, based on econinvent dataset for "market for electricity", for different EU countries (kWh/kg)
- `../data/processed/unit_benefits.csv` - unit benefits of multiple materials, including information on 
    - carbon content (kgCO2)
    - rotation period (yrs)

## Notebook setup 

### Brightway databse setup 

#### Setup and version pinning

In [1]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import importlib.metadata

# ── Version pinning ───────────────────────────────────────────────────────────
for pkg in ["bw2data", "bw2io", "bw2calc"]:
    print(f"  {pkg}: {importlib.metadata.version(pkg)}")
print()

# ── Project and database naming ───────────────────────────────────────────────
PROJECT_NAME     = "biobased_construction_impact_calculator"
EI_VERSION       = "3.12"
EI_MODEL         = "cutoff"
EI_DB_NAME       = f"ecoinvent-{EI_VERSION}-{EI_MODEL}"       # "ecoinvent-3.12-cutoff"
BIOSPHERE_NAME   = f"ecoinvent-{EI_VERSION}-biosphere"        # "ecoinvent-3.12-biosphere"

# ── LCIA method naming ────────────────────────────────────────────────────────
# This ecoinvent version namespaces methods one level deeper than older
# Brightway convention: (namespace, family, category, indicator) instead of
# (family, category, indicator).
METHOD_NAMESPACE = f"ecoinvent-{EI_VERSION}"                  # "ecoinvent-3.12"
METHOD_FAMILY    = "EF v3.1"                                  # not "EF v3.1 no LT"

print(f"Target project:    {PROJECT_NAME}")
print(f"Target database:   {EI_DB_NAME}")
print(f"Target biosphere:  {BIOSPHERE_NAME}")
print(f"Method namespace:  {METHOD_NAMESPACE}")
print(f"Method family:     {METHOD_FAMILY}")

  bw2data: 4.4.2
  bw2io: 0.9.6
  bw2calc: 2.0.1

Target project:    biobased_construction_impact_calculator
Target database:   ecoinvent-3.12-cutoff
Target biosphere:  ecoinvent-3.12-biosphere
Method namespace:  ecoinvent-3.12
Method family:     EF v3.1


#### Create project and import ecoinvent

In [2]:
import os

# ── Connect to (or create) the project ────────────────────────────────────────
bd.projects.set_current(PROJECT_NAME)
print(f"Active project: {bd.projects.current}")

# ── Import ecoinvent, if not already present ──────────────────────────────────
if EI_DB_NAME in bd.databases:
    print(f"'{EI_DB_NAME}' already imported ({len(bd.Database(EI_DB_NAME))} datasets) — skipping.")
else:
    ei_username = os.environ.get("ECOINVENT_USERNAME")
    ei_password = os.environ.get("ECOINVENT_PASSWORD")

    bi.import_ecoinvent_release(
        version=EI_VERSION,
        system_model=EI_MODEL,
        username=ei_username,
        password=ei_password,
    )
    print(f"Imported '{EI_DB_NAME}': {len(bd.Database(EI_DB_NAME))} datasets")

ei = bd.Database(EI_DB_NAME)

Active project: biobased_construction_impact_calculator
'ecoinvent-3.12-cutoff' already imported (26533 datasets) — skipping.


#### Define EF v3.1 impact categories

In [3]:
# ── EF v3.1 methods, namespaced under this ecoinvent version ─────────────────
EF_METHODS = [
    m for m in bd.methods
    if m[0] == METHOD_NAMESPACE and m[1] == METHOD_FAMILY
]
print(f"Found {len(EF_METHODS)} EF v3.1 method tuples")

# ── Units lookup, keyed on method[2] (the category level) ────────────────────
EF_UNITS = {
    'acidification':                                     'mol H+-eq',
    'climate change':                                     'kg CO2-eq',
    'climate change: biogenic':                           'kg CO2-eq',
    'climate change: fossil':                             'kg CO2-eq',
    'climate change: land use and land use change':       'kg CO2-eq',
    'ecotoxicity: freshwater':                            'CTUe',
    'ecotoxicity: freshwater, inorganics':                'CTUe',
    'ecotoxicity: freshwater, organics':                  'CTUe',
    'energy resources: non-renewable':                    'MJ',
    'eutrophication: freshwater':                         'kg P-eq',
    'eutrophication: marine':                             'kg N-eq',
    'eutrophication: terrestrial':                        'mol N-eq',
    'human toxicity: carcinogenic':                       'CTUh',
    'human toxicity: carcinogenic, inorganics':           'CTUh',
    'human toxicity: carcinogenic, organics':             'CTUh',
    'human toxicity: non-carcinogenic':                   'CTUh',
    'human toxicity: non-carcinogenic, inorganics':       'CTUh',
    'human toxicity: non-carcinogenic, organics':         'CTUh',
    'ionising radiation: human health':                   'kBq U235-eq',
    'land use':                                           'dimensionless (soil quality index)',
    'material resources: metals/minerals':                'kg Sb-eq',
    'ozone depletion':                                    'kg CFC-11-eq',
    'particulate matter formation':                       'disease incidence',
    'photochemical oxidant formation: human health':      'dimensionless (tropospheric ozone concentration increase)',
    'water use':                                           'm3 world-eq deprived',
}

missing_units = [m[2] for m in EF_METHODS if m[2] not in EF_UNITS]
if missing_units:
    print(f"⚠ {len(missing_units)} categories have no unit defined: {missing_units}")
else:
    print(f"All {len(EF_METHODS)} categories have a unit defined.")

Found 25 EF v3.1 method tuples
All 25 categories have a unit defined.


### Helper functions 

e.g. ecoinvent database lookup 

In [4]:
def find_ei(name, location, ref_product=None):
    """Look up a single ecoinvent activity by exact name and location."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


def search_ei(keyword, max_results=20):
    """
    Broad keyword search across ecoinvent. Used to verify dataset names
    before hard-coding them in find_ei().
    """
    keyword_lower = keyword.lower()
    results = [a for a in ei if keyword_lower in a['name'].lower()]
    print(f"Found {len(results)} dataset(s) matching '{keyword}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print(f"  unit:     {a.get('unit', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more, not shown")

## Unit burdens

#### load source csvs

In [11]:
import pandas as pd

DATA_DIR = "../../data/data_sources"  # adjust to match your repo structure

def load_burden_table(filename, label):
    df = pd.read_csv(f"{DATA_DIR}/{filename}", na_values=["n/a"])
    df.insert(0, "source_table", label)
    return df

df_materials = pd.concat([
    load_burden_table("materials_biobased_burdens.csv",   "biobased"),
    load_burden_table("materials_binding_burdens.csv",    "binding"),
    load_burden_table("materials_baseline_burdens.csv",   "baseline"),
    load_burden_table("processes_collection_burdens.csv", "collection"),
], ignore_index=True)

for col in ["material_name", "ecoinventDataset_name", "geographicalCoverage", "referenceProduct"]:
    df_materials[col] = df_materials[col].astype("string").str.strip()

print(f"Loaded {len(df_materials)} materials across {df_materials['source_table'].nunique()} source tables")
df_materials.head()

Loaded 21 materials across 4 source tables


,source_table,material_name,unit,material_group,allocation_type,ecoinventDataset_name,geographicalCoverage,referenceProduct,alternative_data_source,comments
0,biobased,Peas,1 kg,Food product,product,market for protein pea,GLO,protein pea,NaN,NaN
1,biobased,Cotton fiber,1 kg,Fiber,product,market for seed-cotton,GLO,seed-cotton,NaN,NaN
2,biobased,Hemp fiber,1 kg,Fiber,product,"decorticated fibre production, hemp",FR,"decorticated fibre, hemp",NaN,NaN
3,biobased,Kenaf fiber,1 kg,Fiber,product,"market for fibre, kenaf",GLO,"fibre, kenaf",NaN,NaN
4,biobased,Jute fiber,1 kg,Fiber,product,"market for fibre, jute",GLO,"fibre, jute",NaN,NaN


#### resolve materials to ecoinvent activities

In [12]:
material_ei_activities = {}
resolution_errors = []

for _, row in df_materials[df_materials["ecoinventDataset_name"].notna()].iterrows():
    print(f"Resolving '{row['material_name']}' → '{row['ecoinventDataset_name']}' | {row['geographicalCoverage']}")
    try:
        material_ei_activities[row["material_name"]] = find_ei(
            name=row["ecoinventDataset_name"],
            location=row["geographicalCoverage"],
            ref_product=row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None,
        )
    except ValueError as e:
        resolution_errors.append((row["material_name"], str(e)))

n_expected = df_materials["ecoinventDataset_name"].notna().sum()
print(f"Resolved {len(material_ei_activities)} / {n_expected} materials")
if resolution_errors:
    print("⚠ Failed lookups:")
    for name, err in resolution_errors:
        print(f"  {name}: {err}")

Resolving 'Peas' → 'market for protein pea' | GLO
Resolving 'Cotton fiber' → 'market for seed-cotton' | GLO
Resolving 'Hemp fiber' → 'decorticated fibre production, hemp' | FR
Resolving 'Kenaf fiber' → 'market for fibre, kenaf' | GLO
Resolving 'Jute fiber' → 'market for fibre, jute' | GLO
Resolving 'Grass fiber' → 'market for grass fibre' | GLO
Resolving 'Bark chips' → 'market for bark chips, green, measured as dry mass' | Europe without Switzerland
Resolving 'Sawdust' → 'market for sawdust, green, collected, measured as dry mass' | Europe without Switzerland
Resolving 'Wood (hard wood, raw)' → 'market for sawnwood, hardwood, raw' | GLO
Resolving 'Wood (soft wood, raw)' → 'market for sawnwood, softwood, raw' | GLO
Resolving 'Mechanical fastener (screws, nails, brackets)' → 'market for steel, low-alloyed, hot rolled' | GLO
Resolving 'Wood glue' → 'market for melamine urea formaldehyde adhesive' | GLO
Resolving 'Fiber glue (epoxy resin)' → 'market for epoxy resin, liquid' | RER
Resolving

#### Pea protein binder — precomputed in biopol_lca

In [13]:
CATEGORY_RENAME_MAP = {"photochemical ozone formation": "photochemical oxidant formation: human health"}

bd.projects.set_current("biopol_lca")
binder_act = bd.get_node(name="pea protein binder production", database="lca_database_3DPrintedBiopol")

binder_scores = {}
for method in [m for m in bd.methods if m[0] == "EF v3.1"]:
    lca = bc.LCA({binder_act: 1}, method)
    lca.lci()
    lca.lcia()
    binder_scores[CATEGORY_RENAME_MAP.get(method[1], method[1])] = lca.score

bd.projects.set_current(PROJECT_NAME)  # switch back before continuing

df_binder = pd.DataFrame([
    {"material_name": "Pea protein binder", "impact_category": cat, "score": score,
     "lca_database": "biopol_lca", "lca_method": "EF v3.1"}
    for cat, score in binder_scores.items()
])
print(f"Computed {len(df_binder)} category scores for the pea protein binder")

Computed 25 category scores for the pea protein binder


#### Burden scores for resolved materials

In [14]:
def get_kg_conversion_factor(act):
    """kg per 1 reference unit — 1.0 for kg-based activities, else derived from the 'wet mass' property."""
    if (act.get("unit") or "").lower() == "kilogram":
        return 1.0
    for exc in act.production():
        wet_mass = exc.get("properties", {}).get("wet mass", {}).get("amount")
        if wet_mass:
            return wet_mass
    return None

transport_name = df_materials.loc[df_materials["source_table"] == "collection", "material_name"].iloc[0]

resolved_scores = []
for name, act in material_ei_activities.items():
    if name == transport_name:
        continue  # handled separately below — tkm-based, not a per-kg material burden

    print(f"Scoring '{name}' → '{act['name']}' | {act['location']} | {act.get('reference product', '—')}")
    factor = get_kg_conversion_factor(act)
    if factor is None:
        print(f"⚠ {name}: no unit conversion available — skipped")
        continue

    for method in EF_METHODS:
        lca = bc.LCA({act: 1}, method)
        lca.lci()
        lca.lcia()
        resolved_scores.append({
            "material_name": name, "impact_category": method[2],
            "score": lca.score / factor,
            "lca_database": method[0], "lca_method": method[1],
        })

df_resolved = pd.DataFrame(resolved_scores)
print(f"Scored {df_resolved['material_name'].nunique()} materials")

Scoring 'Peas' → 'market for protein pea' | GLO | protein pea
Scoring 'Cotton fiber' → 'market for seed-cotton' | GLO | seed-cotton
Scoring 'Hemp fiber' → 'decorticated fibre production, hemp' | FR | decorticated fibre, hemp
Scoring 'Kenaf fiber' → 'market for fibre, kenaf' | GLO | fibre, kenaf
Scoring 'Jute fiber' → 'market for fibre, jute' | GLO | fibre, jute
Scoring 'Grass fiber' → 'market for grass fibre' | GLO | grass fibre
Scoring 'Bark chips' → 'market for bark chips, green, measured as dry mass' | Europe without Switzerland | bark chips, green, measured as dry mass
Scoring 'Sawdust' → 'market for sawdust, green, collected, measured as dry mass' | Europe without Switzerland | sawdust, green, collected, measured as dry mass
Scoring 'Wood (hard wood, raw)' → 'market for sawnwood, hardwood, raw' | GLO | sawnwood, hardwood, raw
Scoring 'Wood (soft wood, raw)' → 'market for sawnwood, softwood, raw' | GLO | sawnwood, softwood, raw
Scoring 'Mechanical fastener (screws, nails, brackets)

#### Transport — scaled to a 1 kg, 50 km basis

In [15]:
TRANSPORT_DISTANCE_KM = 50  # generic collection distance assumption
tkm_per_kg = TRANSPORT_DISTANCE_KM * 0.001  # t·km per kg material

transport_act = material_ei_activities[transport_name]

transport_scores = []
for method in EF_METHODS:
    lca = bc.LCA({transport_act: 1}, method)
    lca.lci()
    lca.lcia()
    transport_scores.append({
        "material_name": transport_name, "impact_category": method[2],
        "score": lca.score * tkm_per_kg,
        "lca_database": method[0], "lca_method": method[1],
    })

df_transport = pd.DataFrame(transport_scores)

#### Assemble and export unit_burdens.csv

In [16]:
material_attrs = (
    df_materials.drop_duplicates("material_name")
    .set_index("material_name")[["material_group", "allocation_type"]]
    .rename(columns={"allocation_type": "typical_allocation_type"})
)

df_burdens_material = pd.concat([df_resolved, df_transport, df_binder], ignore_index=True)
df_burdens_material = df_burdens_material.merge(material_attrs, on="material_name", how="left")
df_burdens_material["impact_category_unit"] = df_burdens_material["impact_category"].map(EF_UNITS)
df_burdens_material["material_amount"] = 1
df_burdens_material["material_unit"] = "kg"

df_burdens_material = df_burdens_material[[
    "material_name", "material_amount", "material_unit",
    "impact_category", "impact_category_unit", "score",
    "lca_database", "lca_method", "material_group", "typical_allocation_type",
]]

n_materials = df_burdens_material["material_name"].nunique()
print(f"Material production/transport: {n_materials} materials × {df_burdens_material['impact_category'].nunique()} categories = {len(df_burdens_material)} rows")

df_burdens_material.head()

Material production/transport: 19 materials × 25 categories = 475 rows


,material_name,material_amount,material_unit,impact_category,impact_category_unit,score,lca_database,lca_method,material_group,typical_allocation_type
0,Peas,1,kg,acidification,mol H+-eq,0.002417,ecoinvent-3.12,EF v3.1,Food product,product
1,Peas,1,kg,climate change,kg CO2-eq,0.409306,ecoinvent-3.12,EF v3.1,Food product,product
2,Peas,1,kg,climate change: biogenic,kg CO2-eq,0.000281,ecoinvent-3.12,EF v3.1,Food product,product
3,Peas,1,kg,climate change: fossil,kg CO2-eq,0.408555,ecoinvent-3.12,EF v3.1,Food product,product
4,Peas,1,kg,climate change: land use and land use change,kg CO2-eq,0.000470,ecoinvent-3.12,EF v3.1,Food product,product


### Machine wear and energy use

In [17]:
# machine wear: unit burden per kg 

MACHINE_NAME     = "market for industrial machine, heavy, unspecified"
MACHINE_LOCATION = "RER"

machine_act = find_ei(MACHINE_NAME, MACHINE_LOCATION)
machine_kg_factor = get_kg_conversion_factor(machine_act)
if machine_kg_factor is None:
    raise ValueError(f"'{machine_act['name']}' has no kg-conversion available — resolve manually.")

print(f"Resolved: '{machine_act['name']}' | {machine_act['location']} | kg-conversion factor: {machine_kg_factor}")

machine_scores = []
for method in EF_METHODS:
    lca = bc.LCA({machine_act: 1}, method)
    lca.lci()
    lca.lcia()
    machine_scores.append({
        "material_name": "Industrial machine (wear)",
        "impact_category": method[2],
        "score": lca.score / machine_kg_factor,
        "lca_database": method[0], "lca_method": method[1],
    })

df_machine = pd.DataFrame(machine_scores)
df_machine["material_group"] = "Machine"
df_machine["typical_allocation_type"] = pd.NA
df_machine["material_amount"] = 1
df_machine["material_unit"] = "kg"

print(f"Machine wear: {len(df_machine)} rows")
df_machine.head()

Resolved: 'market for industrial machine, heavy, unspecified' | RER | kg-conversion factor: 1.0
Machine wear: 25 rows


,material_name,impact_category,score,lca_database,lca_method,material_group,typical_allocation_type,material_amount,material_unit
0,Industrial machine (wear),acidification,0.017537,ecoinvent-3.12,EF v3.1,Machine,<NA>,1,kg
1,Industrial machine (wear),climate change,2.531240,ecoinvent-3.12,EF v3.1,Machine,<NA>,1,kg
2,Industrial machine (wear),climate change: biogenic,0.002060,ecoinvent-3.12,EF v3.1,Machine,<NA>,1,kg
3,Industrial machine (wear),climate change: fossil,2.524260,ecoinvent-3.12,EF v3.1,Machine,<NA>,1,kg
4,Industrial machine (wear),climate change: land use and land use change,0.004920,ecoinvent-3.12,EF v3.1,Machine,<NA>,1,kg


In [18]:
# electricity: unit burden per kWh, all European countries 

ELECTRICITY_ACTIVITY_NAME = "market for electricity, medium voltage"

EUROPEAN_LOCATIONS = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR",
    "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK",
    "SI", "ES", "SE", "GB", "CH", "NO",
]

electricity_activities = {}
missing_locations = []
for loc in EUROPEAN_LOCATIONS:
    try:
        electricity_activities[loc] = find_ei(ELECTRICITY_ACTIVITY_NAME, loc)
    except ValueError:
        missing_locations.append(loc)

print(f"Resolved electricity market for {len(electricity_activities)} / {len(EUROPEAN_LOCATIONS)} countries")
if missing_locations:
    print(f"⚠ No dataset found for: {', '.join(missing_locations)}")

# ── Unit burden per kWh, per country — already on a kWh basis, no conversion ──
electricity_scores = []
for loc, act in electricity_activities.items():
    print(f"Scoring electricity | {loc} ...")
    for method in EF_METHODS:
        lca = bc.LCA({act: 1}, method)
        lca.lci()
        lca.lcia()
        electricity_scores.append({
            "material_name": f"Electricity ({loc})",
            "impact_category": method[2],
            "score": lca.score,
            "lca_database": method[0], "lca_method": method[1],
        })

df_electricity = pd.DataFrame(electricity_scores)
df_electricity["material_group"] = "Energy"
df_electricity["typical_allocation_type"] = pd.NA
df_electricity["material_amount"] = 1
df_electricity["material_unit"] = "kWh"

print(f"Electricity: {df_electricity['material_name'].nunique()} countries × {df_electricity['impact_category'].nunique()} categories = {len(df_electricity)} rows")

Resolved electricity market for 30 / 30 countries
Scoring electricity | AT ...
Scoring electricity | BE ...
Scoring electricity | BG ...
Scoring electricity | HR ...
Scoring electricity | CY ...
Scoring electricity | CZ ...
Scoring electricity | DK ...
Scoring electricity | EE ...
Scoring electricity | FI ...
Scoring electricity | FR ...
Scoring electricity | DE ...
Scoring electricity | GR ...
Scoring electricity | HU ...
Scoring electricity | IE ...
Scoring electricity | IT ...
Scoring electricity | LV ...
Scoring electricity | LT ...
Scoring electricity | LU ...
Scoring electricity | MT ...
Scoring electricity | NL ...
Scoring electricity | PL ...
Scoring electricity | PT ...
Scoring electricity | RO ...
Scoring electricity | SK ...
Scoring electricity | SI ...
Scoring electricity | ES ...
Scoring electricity | SE ...
Scoring electricity | GB ...
Scoring electricity | CH ...
Scoring electricity | NO ...
Electricity: 30 countries × 25 categories = 750 rows


In [19]:
# assemble machine + energy unit burdens 

df_burdens_machine = pd.concat([df_machine, df_electricity], ignore_index=True)
df_burdens_machine["impact_category_unit"] = df_burdens_machine["impact_category"].map(EF_UNITS)

df_burdens_machine = df_burdens_machine[[
    "material_name", "material_amount", "material_unit",
    "impact_category", "impact_category_unit", "score",
    "lca_database", "lca_method", "material_group", "typical_allocation_type",
]]

print(f"Machine + energy: {df_burdens_machine['material_name'].nunique()} items × "
      f"{df_burdens_machine['impact_category'].nunique()} categories = {len(df_burdens_machine)} rows")
df_burdens_machine.head()

Machine + energy: 31 items × 25 categories = 775 rows


,material_name,material_amount,material_unit,impact_category,impact_category_unit,score,lca_database,lca_method,material_group,typical_allocation_type
0,Industrial machine (wear),1,kg,acidification,mol H+-eq,0.017537,ecoinvent-3.12,EF v3.1,Machine,NaN
1,Industrial machine (wear),1,kg,climate change,kg CO2-eq,2.531240,ecoinvent-3.12,EF v3.1,Machine,NaN
2,Industrial machine (wear),1,kg,climate change: biogenic,kg CO2-eq,0.002060,ecoinvent-3.12,EF v3.1,Machine,NaN
3,Industrial machine (wear),1,kg,climate change: fossil,kg CO2-eq,2.524260,ecoinvent-3.12,EF v3.1,Machine,NaN
4,Industrial machine (wear),1,kg,climate change: land use and land use change,kg CO2-eq,0.004920,ecoinvent-3.12,EF v3.1,Machine,NaN


### Assemble burdens and save as csv

In [20]:
OUTPUT_DIR = "../../data/processed"  # adjust to match your repo structure

df_burdens = pd.concat([df_burdens_material, df_burdens_machine], ignore_index=True)

n_items = df_burdens["material_name"].nunique()
n_categories = df_burdens["impact_category"].nunique()
print(f"Unit burdens: {n_items} materials/processes × {n_categories} categories = {len(df_burdens)} rows")

df_burdens.to_csv(f"{OUTPUT_DIR}/unit_burdens.csv", index=False)
print(f"Exported to {OUTPUT_DIR}/unit_burdens.csv")

df_burdens.head()

Unit burdens: 50 materials/processes × 25 categories = 1250 rows
Exported to ../../data/processed/unit_burdens.csv


,material_name,material_amount,material_unit,impact_category,impact_category_unit,score,lca_database,lca_method,material_group,typical_allocation_type
0,Peas,1,kg,acidification,mol H+-eq,0.002417,ecoinvent-3.12,EF v3.1,Food product,product
1,Peas,1,kg,climate change,kg CO2-eq,0.409306,ecoinvent-3.12,EF v3.1,Food product,product
2,Peas,1,kg,climate change: biogenic,kg CO2-eq,0.000281,ecoinvent-3.12,EF v3.1,Food product,product
3,Peas,1,kg,climate change: fossil,kg CO2-eq,0.408555,ecoinvent-3.12,EF v3.1,Food product,product
4,Peas,1,kg,climate change: land use and land use change,kg CO2-eq,0.000470,ecoinvent-3.12,EF v3.1,Food product,product


## Unit benefit parameters 

Extract `carbon content` and `rotation period` values for biobased materials based on literature and ecoinvent. Because the benefits (carbon sequestration) depends on both product mass *and* lifespan, we are calculating the *parameters* for calculating benefits, not the benfits themselves. These can only be calculated when we know what the product lifespan is, and this is only defined in `02_model.ipynb`. 

**Important note**: the unit for carbon content is kg carbon per kg material, not kg CO2 per kg material. So when calculating carbon sequestration in the model (next notebook `02_model.ipynb`), we first need to multiple this carbon content value by the molecular weight ratio between CO2 and C, which is about 3.67. 

#### Pea protein binder — derived carbon fraction

Pea protein binder isn't pure pea — it's derived from the LCI chain
(`peas → AEIEP pea protein isolate production → pea protein binder production`)
in `biopol_lca`, same cross-project pattern as the burden calculation. The pea
mass fraction scales the pure-pea carbon fraction down to a binder-level value.

In [21]:
def get_exchange_amount(node, producer_name_contains):
    """Amount of a technosphere exchange, matched by (partial) producer name."""
    matches = [e for e in node.technosphere() if producer_name_contains in e.input['name']]
    if not matches:
        raise ValueError(f"No exchange found containing '{producer_name_contains}'")
    return matches[0]['amount']

bd.projects.set_current("biopol_lca")
node_isolate = bd.get_node(name="AEIEP pea protein isolate production", database="lca_database_3DPrintedBiopol")
node_binder  = bd.get_node(name="pea protein binder production",        database="lca_database_3DPrintedBiopol")

kg_peas_per_kg_isolate   = get_exchange_amount(node_isolate, "protein pea")
kg_isolate_per_kg_binder = get_exchange_amount(node_binder, "AEIEP pea protein isolate production")
kg_peas_per_kg_binder    = kg_peas_per_kg_isolate * kg_isolate_per_kg_binder

bd.projects.set_current(PROJECT_NAME)  # switch back before continuing

CARBON_FRACTION_PEA = 0.45  # pure whole dried pea, IPCC Tier 1 / Lal (2004)
carbon_fraction_binder = CARBON_FRACTION_PEA * kg_peas_per_kg_binder

print(f"kg peas / kg binder:            {kg_peas_per_kg_binder:.4f}")
print(f"Derived binder carbon fraction: {carbon_fraction_binder:.4f}")

kg peas / kg binder:            0.5893
Derived binder carbon fraction: 0.2652


#### Extract carbon content data

In [22]:
import pandas as pd

# resolve biobased benefits to ecoinvent datasets, for later scoring

DATA_DIR = "../../data/data_sources"  # adjust to match your repo structure

# ── Reload the updated benefits CSV ───────────────────────────────────────────
df_benefits = pd.read_csv(f"{DATA_DIR}/materials_biobased_benefits.csv", na_values=["n/a"])
for col in ["material_name", "ecoinventDataset_name", "geographicalCoverage", "referenceProduct"]:
    df_benefits[col] = df_benefits[col].astype("string").str.strip()

# ── Resolve each material with an ecoinvent dataset ───────────────────────────
benefit_ei_activities = {}
benefit_resolution_errors = []

resolvable_benefits = df_benefits[df_benefits["ecoinventDataset_name"].notna()]
no_ecoinvent = df_benefits[df_benefits["ecoinventDataset_name"].isna()]

for _, row in resolvable_benefits.iterrows():
    print(f"Resolving benefit '{row['material_name']}' → '{row['ecoinventDataset_name']}' | {row['geographicalCoverage']} ... ")
    try:
        act = find_ei(
            name=row["ecoinventDataset_name"],
            location=row["geographicalCoverage"],
            ref_product=row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None,
        )
        benefit_ei_activities[row["material_name"]] = act
    except ValueError as e:
        benefit_resolution_errors.append((row["material_name"], str(e)))

print(f"Resolved:  {len(benefit_ei_activities)} / {len(resolvable_benefits)}")
print(f"No ecoinvent dataset (handled separately): {no_ecoinvent['material_name'].tolist()}")
if benefit_resolution_errors:
    print(f"\n⚠ {len(benefit_resolution_errors)} lookup(s) FAILED:")
    for name, err in benefit_resolution_errors:
        print(f"  {name}: {err}")

Resolving benefit 'Peas' → 'market for protein pea' | GLO ... 
Resolving benefit 'Cotton fiber' → 'market for seed-cotton' | GLO ... 
Resolving benefit 'Hemp fiber' → 'decorticated fibre production, hemp' | FR ... 
Resolving benefit 'Kenaf fiber' → 'market for fibre, kenaf' | GLO ... 
Resolving benefit 'Jute fiber' → 'market for fibre, jute' | GLO ... 
Resolving benefit 'Grass fiber' → 'market for grass fibre' | GLO ... 
Resolving benefit 'Bark chips' → 'market for bark chips, green, measured as dry mass' | Europe without Switzerland ... 
Resolving benefit 'Sawdust' → 'market for sawdust, green, collected, measured as dry mass' | Europe without Switzerland ... 
Resolving benefit 'Wood (hard wood, raw)' → 'market for sawnwood, hardwood, raw' | GLO ... 
Resolving benefit 'Wood (soft wood, raw)' → 'market for sawnwood, softwood, raw' | GLO ... 
Resolving benefit 'Cellulose reject fibers' → 'market for waste paper, sorted' | GLO ... 
Resolved:  11 / 11
No ecoinvent dataset (handled separat

In [23]:
# extrack raw properties 

def get_production_properties(act):
    """Return the properties dict from an activity's production exchange, or {} if none."""
    for exc in act.production():
        return exc.get("properties", {})
    return {}

material_properties = {}
for material_name, act in benefit_ei_activities.items():
    props = get_production_properties(act)
    material_properties[material_name] = props
    print(f"{material_name}  ({act['name']} | {act['location']})")
    if props:
        for key, val in props.items():
            print(f"    {key}: {val.get('amount')}")
    else:
        print("    ⚠ no properties found on production exchange")
    print()

Peas  (market for protein pea | GLO)
    carbon allocation: 0.407697116980715
    carbon content: 0.4694207145270348
    carbon content, fossil: 0.0
    carbon content, non-fossil: 0.4694207145270348
    dry mass: 0.8685111337523537
    energy content: 16.077599999999993
    price: 2.109999999999999
    water content: 0.15141521818690692
    water in wet mass: 0.13148886624764583
    wet mass: 0.9999999999999996

Cotton fiber  (market for seed-cotton | GLO)
    carbon allocation: 0.4384249999999996
    carbon content: 0.47499999999999976
    carbon content, fossil: 0.0
    carbon content, non-fossil: 0.47499999999999976
    dry mass: 0.9229999999999997
    price: 0.41101899999999986
    water content: 0.08342361863488616
    water in wet mass: 0.07699999999999996
    wet mass: 0.9999999999999996

Hemp fiber  (decorticated fibre production, hemp | FR)
    allocation factor: 0.606955220812199
    carbon allocation: 0.445704
    carbon content: 0.4548
    carbon content, fossil: 0.0
    c

### Compute carbon content per kg product

For every material with an ecoinvent activity (from 4.1/4.2), applies:

$$
\text{carbon content per kg product} = \text{carbon content, non-fossil} \times \frac{\text{dry mass}}{\text{wet mass}}
$$

Dividing by `wet mass` (rather than assuming it's 1, as the earlier draft did) makes this
correct regardless of the activity's actual reference unit — the same generalisation
that fixed Section 3.1's wood/concrete unit issue applies here too, since these are the
same underlying properties.

**Seagrass** has no ecoinvent activity, so it's handled separately: its carbon content
stays the literal literature value already in `materials_biobased_benefits.csv` (0.336,
dry-mass basis) — still not converted to a wet basis, the same open gap flagged earlier
and still unresolved.

In [24]:
# ── Compute carbon content per kg product for all ecoinvent-resolved materials ──
carbon_content_rows = []

for material_name, props in material_properties.items():
    carbon_nonfossil = props.get("carbon content, non-fossil", {}).get("amount")
    dry_mass = props.get("dry mass", {}).get("amount")
    wet_mass = props.get("wet mass", {}).get("amount")

    if carbon_nonfossil is None or dry_mass is None or wet_mass is None:
        print(f"⚠ {material_name}: missing one of carbon content/dry mass/wet mass — skipped")
        continue

    carbon_per_kg = carbon_nonfossil * (dry_mass / wet_mass)
    carbon_content_rows.append({
        "material_name": material_name,
        "carbon_content_kgC_per_kg": carbon_per_kg,
        "carbon_content_source": "ecoinvent (carbon content, non-fossil × dry mass / wet mass)",
    })

carbon_content_rows.append({
    "material_name": "Pea protein binder",
    "carbon_content_kgC_per_kg": carbon_fraction_binder,
    "carbon_content_source": "derived from biopol_lca LCI chain (peas → isolate → binder), CARBON_FRACTION_PEA=0.45 (Lal, 2004)",
})

df_carbon_ecoinvent = pd.DataFrame(carbon_content_rows)

# ── Seagrass — literal literature value, no ecoinvent activity ───────────────
seagrass_row = df_benefits[df_benefits["material_name"] == "Seagrass"].iloc[0]
df_carbon_seagrass = pd.DataFrame([{
    "material_name": "Seagrass",
    "carbon_content_kgC_per_kg": seagrass_row["carbonContent_dryWeight_kg"],
    "carbon_content_source": seagrass_row["carbonContent_source"] + " [dry-mass basis, NOT converted to wet basis — open gap]",
}])

df_carbon_content = pd.concat([df_carbon_ecoinvent, df_carbon_seagrass], ignore_index=True)

print(f"Computed carbon content for {len(df_carbon_content)} materials:\n")
df_carbon_content

Computed carbon content for 13 materials:



,material_name,carbon_content_kgC_per_kg,carbon_content_source
0,Peas,0.407697,"ecoinvent (carbon content, non-fossil × dry ma..."
1,Cotton fiber,0.438425,"ecoinvent (carbon content, non-fossil × dry ma..."
2,Hemp fiber,0.445704,"ecoinvent (carbon content, non-fossil × dry ma..."
3,Kenaf fiber,0.418944,"ecoinvent (carbon content, non-fossil × dry ma..."
4,Jute fiber,0.404396,"ecoinvent (carbon content, non-fossil × dry ma..."
5,Grass fiber,0.45034,"ecoinvent (carbon content, non-fossil × dry ma..."
6,Bark chips,0.205833,"ecoinvent (carbon content, non-fossil × dry ma..."
7,Sawdust,0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."
8,"Wood (hard wood, raw)",0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."
9,"Wood (soft wood, raw)",0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."


### save as csv

In [31]:
OUTPUT_DIR = "../../data/processed"  # adjust to match your repo structure

# ── Merge carbon content + rotation period + material_group ──────────────────
df_benefit_params = df_benefits[["material_name", "rotationPeriod_yr", "rotationPeriod_source"]].rename(
    columns={"rotationPeriod_yr": "rotation_period_yr", "rotationPeriod_source": "rotation_period_source"}
)

# ── Add pea protein binder as its own row before merging ─────────────────────
df_benefit_params_binder = pd.DataFrame([{
    "material_name": "Pea protein binder",
    "rotation_period_yr": 1,
    "rotation_period_source": "Annual crop, same as peas",
}])
df_benefit_params = pd.concat([df_benefit_params, df_benefit_params_binder], ignore_index=True)

df_benefit_params = (
    df_benefit_params
    .merge(df_carbon_content, on="material_name", how="left")
    .merge(material_attrs[["material_group"]], left_on="material_name", right_index=True, how="left")
)

df_benefit_params = df_benefit_params[[
    "material_name", "material_group",
    "carbon_content_kgC_per_kg", "carbon_content_source",
    "rotation_period_yr", "rotation_period_source",
]]

# ── QA: every biobased material should be covered ─────────────────────────────
all_biobased_materials = set(df_materials[df_materials["source_table"] == "biobased"]["material_name"].unique())
benefit_materials = set(df_benefit_params["material_name"].unique())
missing = all_biobased_materials - benefit_materials

if missing:
    print(f"⚠ {len(missing)} biobased material(s) missing from benefit params: {sorted(missing)}")
else:
    print(f"All {len(all_biobased_materials)} biobased materials covered.")

missing_carbon = df_benefit_params[df_benefit_params["carbon_content_kgC_per_kg"].isna()]["material_name"].tolist()
if missing_carbon:
    print(f"⚠ {len(missing_carbon)} material(s) with no carbon content value: {missing_carbon}")

# ── Export ─────────────────────────────────────────────────────────────────────
df_benefit_params.to_csv(f"{OUTPUT_DIR}/unit_benefit_params.csv", index=False)
print(f"\nWritten to {OUTPUT_DIR}/unit_benefit_params.csv")

df_benefit_params

All 12 biobased materials covered.

Written to ../../data/processed/unit_benefit_params.csv


,material_name,material_group,carbon_content_kgC_per_kg,carbon_content_source,rotation_period_yr,rotation_period_source
0,Peas,Food product,0.407697,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
1,Cotton fiber,Fiber,0.438425,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
2,Hemp fiber,Fiber,0.445704,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
3,Kenaf fiber,Fiber,0.418944,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
4,Jute fiber,Fiber,0.404396,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
5,Grass fiber,Fiber,0.45034,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
6,Bark chips,Forestry by-product,0.205833,"ecoinvent (carbon content, non-fossil × dry ma...",80.0,average between soft and hard wood
7,Sawdust,Forestry by-product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",80.0,average between soft and hard wood
8,"Wood (hard wood, raw)",Forestry product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",100.0,"Beech rotation (110–140 yr, or 80–100 yr in so..."
9,"Wood (soft wood, raw)",Forestry product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",65.0,Softwood CORRIM figure: Cradle-to-gate LCA of ...
